# Training Table Walkthrough

This notebook explains and audits the pass-detection training table created from the Step 2 master join table.

Expected contract:

- the master join is split by match before feature engineering
- each output row is one non-overlapping 5-frame window, about 0.5 seconds
- train, validation, and test are written as separate Parquet files
- the target is `is_pass`, derived from `primary_event == "PASS"`
- windows where all ball position values are missing are skipped
- raw features are calculated in interpretable units first
- continuous features are standardized with a train-fitted scaler only after the table is built
- event coordinate columns from the master join are excluded to avoid leakage

Plain-language companion doc: `docs/training_table_walkthrough.md`.

## 1. Setup

This cell makes the notebook work whether it is opened from the project root or from the `notebooks/` folder.

In [1]:
from pathlib import Path
import importlib
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

from driblab.config import CONFIG_PATH, MODEL_BASE_DATA_DIR
from driblab.features import match_splits
from driblab.features import training_table as training_table_module

training_table_module = importlib.reload(training_table_module)
OUTPUT_COLUMNS = training_table_module.OUTPUT_COLUMNS
CONTINUOUS_FEATURES = training_table_module.CONTINUOUS_FEATURES

MASTER_JOIN_PATH = MODEL_BASE_DATA_DIR / "master_join_table.parquet"
TRAINING_PATHS = {
    "train": MODEL_BASE_DATA_DIR / "training_table_train.parquet",
    "validation": MODEL_BASE_DATA_DIR / "training_table_validation.parquet",
    "test": MODEL_BASE_DATA_DIR / "training_table_test.parquet",
}
SUMMARY_PATHS = {
    split: MODEL_BASE_DATA_DIR / f"training_table_summary_{split}.csv"
    for split in TRAINING_PATHS
}

print(PROJECT_ROOT)

/Users/nataliaurrea/Documents/IE/MBDS/Term III/Driblab


## 2. Rebuild the training tables

Run this from a terminal at the project root when you want to regenerate the files:

```bash
conda activate driblabvenv
python -m driblab.features.training_table
```

The command reads `master_join_table.parquet` and `config.yaml`, builds raw window features first, writes one table and one summary CSV per split, then standardizes continuous features and saves `artifacts/models/feature_scaler.pkl`.

In [2]:
assert MASTER_JOIN_PATH.exists(), f"Missing {MASTER_JOIN_PATH}. Run Step 2 first."
for split, path in TRAINING_PATHS.items():
    assert path.exists(), f"Missing {path}. Run: python -m driblab.features.training_table"
for split, path in SUMMARY_PATHS.items():
    assert path.exists(), f"Missing {path}. Run the training table builder."

file_inventory = pd.DataFrame(
    [
        {
            "split": split,
            "table_path": str(TRAINING_PATHS[split].relative_to(PROJECT_ROOT)),
            "table_mb": TRAINING_PATHS[split].stat().st_size / 1024 / 1024,
            "summary_path": str(SUMMARY_PATHS[split].relative_to(PROJECT_ROOT)),
        }
        for split in TRAINING_PATHS
    ]
)
display(file_inventory)

,split,table_path,table_mb,summary_path
0,train,data/processed/model_base/training_table_train...,10.611863,data/processed/model_base/training_table_summa...
1,validation,data/processed/model_base/training_table_valid...,2.872941,data/processed/model_base/training_table_summa...
2,test,data/processed/model_base/training_table_test....,2.674035,data/processed/model_base/training_table_summa...


## 3. Load summaries

The summary CSVs are the quickest health check. They count windows, pass windows, no-event windows, match coverage, and period coverage.

In [3]:
summaries = pd.concat(
    [pd.read_csv(path) for path in SUMMARY_PATHS.values()],
    ignore_index=True,
)
summaries["pass_percentage"] = summaries["pass_percentage"].round(2)
display(summaries)

,split_name,total_windows,pass_windows,no_event_windows,pass_percentage,unique_matches,unique_periods
0,train,161505,15554,140892,9.63,23,2
1,validation,36719,3567,32044,9.71,5,2
2,test,35085,3725,30251,10.62,5,2


## 4. Check schemas

All split tables should have the same 25 columns, in the same order.

In [4]:
schema_checks = []
for split, path in TRAINING_PATHS.items():
    columns = pq.ParquetFile(path).schema_arrow.names
    leakage_columns = [
        column
        for column in ["e.x_meters_absolute", "e.y_meters_absolute"]
        if column in columns
    ]
    schema_checks.append(
        {
            "split": split,
            "column_count": len(columns),
            "matches_expected_order": columns == OUTPUT_COLUMNS,
            "leakage_columns_present": ", ".join(leakage_columns) if leakage_columns else "none",
        }
    )

schema_checks = pd.DataFrame(schema_checks)
display(schema_checks)
display(pd.DataFrame({"column_order": OUTPUT_COLUMNS}))

scaler_path = PROJECT_ROOT / "artifacts/models/feature_scaler.pkl"
assert scaler_path.exists(), f"Missing scaler: {scaler_path}"

train_features = pd.read_parquet(
    TRAINING_PATHS["train"],
    columns=CONTINUOUS_FEATURES,
)
normalization_check = pd.DataFrame(
    {
        "continuous_nan_count": [
            int(train_features.isna().sum().sum())
        ],
        "leakage_columns_absent": [
            bool((schema_checks["leakage_columns_present"] == "none").all())
        ],
    }
)
display(normalization_check)

assert len(OUTPUT_COLUMNS) == 23
assert schema_checks["matches_expected_order"].all()
assert normalization_check.loc[0, "continuous_nan_count"] == 0
assert normalization_check.loc[0, "leakage_columns_absent"]


,split,column_count,matches_expected_order,leakage_columns_present
0,train,23,True,none
1,validation,23,True,none
2,test,23,True,none


,column_order
0,t.match_id
1,t.period
2,window_time
3,data_split
4,is_attacking_direction
5,primary_event
6,is_pass
7,secondary_events
8,ball_x_avg
9,ball_y_avg


,continuous_nan_count,leakage_columns_absent
0,0,True


## 5. Raw pre-standardization preview

The saved training tables are model-ready, so their continuous columns have already been standardized. To keep the feature engineering interpretable, this cell rebuilds a small raw preview directly from `master_join_table.parquet` using the same `training_table.py` functions, but without applying `StandardScaler`.

This preview shows the units before scaling: tracking meters, pitch meters, counts, and binary flags.

In [5]:
raw_master_join, _ = training_table_module.load_training_inputs(
    MASTER_JOIN_PATH,
    CONFIG_PATH,
)

preview_keys = (
    raw_master_join.loc[
        raw_master_join["data_split"] == "train",
        ["t.match_id", "t.period"],
    ]
    .drop_duplicates()
    .head(2)
)
raw_preview_source = raw_master_join.merge(
    preview_keys,
    on=["t.match_id", "t.period"],
    how="inner",
)
raw_preview_table = training_table_module.build_training_table_for_split(
    raw_preview_source,
    "train",
)

display(raw_preview_table[OUTPUT_COLUMNS].head(10))



Skipped 5329 train windows with all ball position values missing.


,t.match_id,t.period,window_time,data_split,is_attacking_direction,primary_event,is_pass,secondary_events,ball_x_avg,ball_y_avg,ball_z_avg,ball_speed_avg,ball_speed_change,ball_direction_x,ball_direction_y,closest_player_dist_start,closest_player_team_start,closest_player_dist_end,closest_player_team_end,closest_player_dist_change,n_players_near_ball,n_unique_players_in_frame,team_changed
0,678949,1,3.0,train,1,no event,0,,41.6125,42.5025,0.0165,1.278106,-0.482571,NaN,NaN,NaN,unknown,2.712803,1925,NaN,1,14,0
1,678949,1,3.5,train,1,no event,0,,37.0460,44.2500,0.1382,1.276188,0.291111,-4.36,2.21,2.973567,1925,7.147195,1925,4.173628,1,14,0
2,678949,1,4.0,train,1,no event,0,,32.2460,46.1860,0.1810,1.196294,1.294278,-3.32,-1.98,8.389923,1925,11.704375,1925,3.314453,0,15,0
3,678949,1,4.5,train,1,no event,0,,27.1880,42.1320,0.0308,0.759206,-0.995796,-2.81,0.08,12.786407,1925,15.089559,1925,2.303152,0,11,0
4,678949,1,5.0,train,1,PASS,1,,24.5520,41.7680,0.0000,0.493233,-0.399088,-0.95,-0.56,15.644175,1925,16.024085,1925,0.379910,0,10,0
5,678949,1,5.5,train,1,no event,0,,24.3040,41.3620,0.0000,0.248368,-0.375047,-0.89,-0.34,15.635527,1925,16.120934,1925,0.485407,0,10,0
6,678949,1,6.0,train,1,no event,0,,23.8900,40.7460,0.0000,0.304405,0.149426,-0.16,-0.63,16.047891,1925,15.963161,1925,-0.084730,0,9,0
7,678949,1,6.5,train,1,no event,0,,23.2740,40.0860,0.0000,0.416254,-0.448219,-1.25,-1.09,15.732962,1946,13.695784,1946,-2.037178,0,8,0
8,678949,1,7.0,train,1,no event,0,,23.0260,40.3140,0.0046,0.975990,-0.092387,-0.24,-0.39,13.354557,1946,11.392910,1946,-1.961647,0,7,0
9,678949,1,7.5,train,1,no event,0,,24.2660,40.1120,0.1162,0.846250,0.985325,2.11,-2.23,11.048552,1946,6.822881,1946,-4.225671,0,5,0


## 6. Saved standardized preview rows

A few rows from each split show the modelling grain: match, period, window time, primary event, target, ball features, closest-player features, player-density features, and the team-change signal. In the default saved tables, continuous numeric features are already standardized; see the final feature notes for the raw pre-standardization definitions.

In [6]:
for split, path in TRAINING_PATHS.items():
    print(split)
    df_preview = pd.read_parquet(path, columns=OUTPUT_COLUMNS).head(5)
    display(df_preview)


train


,t.match_id,t.period,window_time,data_split,is_attacking_direction,primary_event,is_pass,secondary_events,ball_x_avg,ball_y_avg,ball_z_avg,ball_speed_avg,ball_speed_change,ball_direction_x,ball_direction_y,closest_player_dist_start,closest_player_team_start,closest_player_dist_end,closest_player_team_end,closest_player_dist_change,n_players_near_ball,n_unique_players_in_frame,team_changed
0,678949,1,3.0,train,1,no event,0,,-0.036409,-0.009347,-0.338020,-0.018366,-0.003267,-0.001086,0.001742,-0.035352,unknown,-0.028998,1925,-0.000610,0.189225,0.066286,0
1,678949,1,3.5,train,1,no event,0,,-0.058672,-0.006499,-0.332237,-0.018375,0.008025,-0.064534,0.028024,-0.028205,1925,-0.018094,1925,0.038694,0.189225,0.066286,0
2,678949,1,4.0,train,1,no event,0,,-0.082073,-0.003343,-0.330203,-0.018768,0.022665,-0.049399,-0.021806,-0.015186,1925,-0.006888,1925,0.030603,-0.598832,0.297534,0
3,678949,1,4.5,train,1,no event,0,,-0.106732,-0.009951,-0.337340,-0.020918,-0.010757,-0.041978,0.002693,-0.004618,1925,0.001436,1925,0.021079,-0.598832,-0.627455,0
4,678949,1,5.0,train,1,PASS,1,,-0.119583,-0.010544,-0.338804,-0.022227,-0.002048,-0.014911,-0.004918,0.002251,1925,0.003734,1925,0.002968,-0.598832,-0.858702,0


validation


,t.match_id,t.period,window_time,data_split,is_attacking_direction,primary_event,is_pass,secondary_events,ball_x_avg,ball_y_avg,ball_z_avg,ball_speed_avg,ball_speed_change,ball_direction_x,ball_direction_y,closest_player_dist_start,closest_player_team_start,closest_player_dist_end,closest_player_team_end,closest_player_dist_change,n_players_near_ball,n_unique_players_in_frame,team_changed
0,689526,1,0.5,validation,1,PASS,1,,0.009772,-0.070614,-0.105197,0.012346,0.003776,-0.001086,0.001742,-0.035352,unknown,-0.035669,unknown,-0.000610,-0.598832,-3.171175,0
1,689526,1,1.0,validation,1,no event,0,,-0.050011,-0.050015,0.495142,-0.008947,-0.035634,-0.001086,0.001742,-0.035352,unknown,-0.026376,7194,-0.000610,0.189225,1.222523,0
2,689526,1,1.5,validation,1,no event,0,,-0.059140,-0.023101,1.279695,0.133498,-0.041610,-0.017530,0.053712,-0.023959,7194,-0.034331,7194,-0.040121,1.765339,1.222523,0
3,689526,1,2.0,validation,1,no event,0,,-0.055522,-0.008872,1.694529,0.102820,-0.269606,0.162770,-0.061170,-0.028883,7194,-0.023062,7186,0.022324,0.977282,1.222523,1
4,689526,1,2.5,validation,1,no event,0,,0.008797,0.021431,2.577897,0.048721,0.003776,-0.001086,0.001742,-0.035352,unknown,-0.035669,unknown,-0.000610,-0.598832,1.222523,0


test


,t.match_id,t.period,window_time,data_split,is_attacking_direction,primary_event,is_pass,secondary_events,ball_x_avg,ball_y_avg,ball_z_avg,ball_speed_avg,ball_speed_change,ball_direction_x,ball_direction_y,closest_player_dist_start,closest_player_team_start,closest_player_dist_end,closest_player_team_end,closest_player_dist_change,n_players_near_ball,n_unique_players_in_frame,team_changed
0,713910,1,0.5,test,1,no event,0,,0.069989,-0.014252,-0.297798,0.166120,0.003607,-0.001086,0.001742,-0.035352,unknown,-0.035669,unknown,-0.000610,-0.598832,-3.171175,0
1,713910,1,1.0,test,1,no event,0,,0.048477,0.036813,-0.306322,0.025364,-0.563754,-0.075302,0.468171,-0.028474,7188,0.020251,7188,0.186597,0.189225,1.222523,0
2,713910,1,1.5,test,1,no event,0,,0.044177,0.049393,-0.310057,-0.020058,0.012232,0.013757,-0.011935,0.019573,7188,0.018751,7188,-0.007385,-0.598832,1.222523,0
3,713910,1,2.0,test,1,PASS,1,PASS,0.097024,-0.064275,-0.309810,0.023190,-0.560797,0.075312,-0.454936,-0.027188,7188,0.011894,7188,0.149554,0.189225,1.222523,0
4,713910,1,2.5,test,1,no event,0,,0.101939,-0.076995,-0.301267,-0.024653,0.003776,-0.001086,0.001742,-0.035352,unknown,0.011839,7188,-0.000610,-0.598832,1.222523,0


## 6. Confirm split isolation

The training table should contain only matches assigned to that split in `config.yaml`. This is the key leakage-prevention check.

In [7]:
splits = match_splits.load_match_splits(CONFIG_PATH)
split_checks = []

for split, path in TRAINING_PATHS.items():
    df = pd.read_parquet(path, columns=["t.match_id", "data_split"])
    expected_matches = set(map(str, splits[split]))
    actual_matches = set(df["t.match_id"].astype(str).unique())
    split_checks.append(
        {
            "split": split,
            "all_rows_have_expected_label": bool((df["data_split"] == split).all()),
            "actual_matches": len(actual_matches),
            "expected_matches": len(expected_matches),
            "unexpected_matches": sorted(actual_matches - expected_matches),
            "missing_matches": sorted(expected_matches - actual_matches),
        }
    )

display(pd.DataFrame(split_checks))

,split,all_rows_have_expected_label,actual_matches,expected_matches,unexpected_matches,missing_matches
0,train,True,23,23,[],[]
1,validation,True,5,5,[],[]
2,test,True,5,5,[],[]


## 7. Target balance by split

The pass rate should be checked before modelling. Here it is similar across train, validation, and test, which is helpful for evaluation.

In [8]:
target_rows = []
for split, path in TRAINING_PATHS.items():
    df = pd.read_parquet(path, columns=["primary_event", "is_pass"])
    target_rows.append(
        {
            "split": split,
            "windows": len(df),
            "pass_windows": int(df["is_pass"].sum()),
            "pass_percentage": round(df["is_pass"].mean() * 100, 2),
            "no_event_windows": int((df["primary_event"] == "no event").sum()),
        }
    )

display(pd.DataFrame(target_rows))

,split,windows,pass_windows,pass_percentage,no_event_windows
0,train,161505,15554,9.63,140892
1,validation,36719,3567,9.71,32044
2,test,35085,3725,10.62,30251


## 8. Event distribution

`PASS` becomes the positive class. Other selected events and `no event` become negative examples.

In [9]:
event_distribution = []
for split, path in TRAINING_PATHS.items():
    events = pd.read_parquet(path, columns=["primary_event"])
    counts = events["primary_event"].value_counts().rename_axis("primary_event")
    split_counts = counts.reset_index(name="windows")
    split_counts.insert(0, "split", split)
    event_distribution.append(split_counts)

event_distribution = pd.concat(event_distribution, ignore_index=True)
display(event_distribution)

,split,primary_event,windows
0,train,no event,140892
1,train,PASS,15554
2,train,BALL TOUCH,2238
3,train,TACKLE,741
4,train,BALL RECOVERY,646
5,train,AERIAL,605
6,train,TAKEON,501
7,train,FOUL,328
8,validation,no event,32044
9,validation,PASS,3567


## 9. Audit the confirmed skip rule

The builder skips a 5-frame window only when all `t.ball_x`, `t.ball_y`, and `t.ball_z` values are missing across the whole window.

This cell estimates how many complete 5-frame windows existed in the master join after split assignment, then compares that with the generated output row counts.

In [10]:
master_cols = [
    "t.match_id",
    "t.period",
    "t.frame",
    "t.ball_x",
    "t.ball_y",
    "t.ball_z",
]
master = pd.read_parquet(MASTER_JOIN_PATH, columns=master_cols)
master = match_splits.add_data_split_column(
    master,
    splits,
    match_col="t.match_id",
)
master = master.sort_values(["t.match_id", "t.period", "t.frame"])

skip_audit_rows = []
for split, split_df in master.groupby("data_split"):
    if split not in TRAINING_PATHS:
        continue
    complete_windows = 0
    skipped_all_ball_missing = 0
    for _, period_df in split_df.groupby(["t.match_id", "t.period"], sort=False):
        complete_rows = (len(period_df) // 5) * 5
        if complete_rows == 0:
            continue
        ball = period_df.iloc[:complete_rows][["t.ball_x", "t.ball_y", "t.ball_z"]]
        ball = ball.apply(pd.to_numeric, errors="coerce").to_numpy().reshape(-1, 5, 3)
        complete_windows += len(ball)
        skipped_all_ball_missing += int(pd.isna(ball).all(axis=(1, 2)).sum())

    output_windows = int(summaries.loc[summaries["split_name"] == split, "total_windows"].iloc[0])
    skip_audit_rows.append(
        {
            "split": split,
            "complete_5_frame_windows": complete_windows,
            "skipped_all_ball_missing": skipped_all_ball_missing,
            "output_windows": output_windows,
            "complete_minus_skipped": complete_windows - skipped_all_ball_missing,
            "matches_output_count": complete_windows - skipped_all_ball_missing == output_windows,
        }
    )

display(pd.DataFrame(skip_audit_rows))

,split,complete_5_frame_windows,skipped_all_ball_missing,output_windows,complete_minus_skipped,matches_output_count
0,test,58488,23403,35085,35085,True
1,train,279437,117932,161505,161505,True
2,validation,59372,22653,36719,36719,True


## 10. Feature notes

- Raw features are built first, before standardization removes the original units.
- `ball_x_avg`, `ball_y_avg`, and `ball_z_avg` start as raw tracking-coordinate averages in meters.
- `ball_speed_avg` starts as the mean valid 3D frame-to-frame ball movement.
- `ball_speed_change` starts as last valid ball movement minus first valid ball movement.
- `ball_direction_x` and `ball_direction_y` start as frame-1-to-frame-5 differences.
- Closest-player distances start as raw tracking x/y meter distances.
- `closest_player_dist_change` starts as end closest-player distance minus start closest-player distance.
- `n_players_near_ball` starts as the count of unique visible player IDs within 5 meters of the ball at least once in the window.
- `n_unique_players_in_frame` starts as the count of unique visible player IDs tracked anywhere in the window.
- `team_changed` is a model feature, not the label.
- `primary_event` is selected by smallest `nearest_timestamp_distance_sec` inside the window.
- `secondary_events` stores the other event types from the same window.
- Missing numeric feature values are kept unless the whole window has no ball position at all.
- Standardization is the final step: `StandardScaler` is fit on train only, then applied to train, validation, and test.
- Missing continuous values are filled with `0` before the scaler transform.
- Event coordinate columns from the master join are not included in the training table or model features.


## 11. Next modelling step

The generated tables are ready to feed into a binary pass classifier. A first baseline can use the numeric columns plus encoded categorical columns for closest-player teams and primary/secondary event context, while keeping the split files separate for evaluation.